# Multi-Step Agent Repair: Which Restart Strategy Wins?

**AAAI'27 Experiment Pipeline — Colab A100 + Qwen3.6-27B-AWQ**

| Stage | What it does | ~Time (A100-40GB, 500 Qs) |
|---|---|---|
| 0 | Download HotpotQA, create folders | 1 min |
| 1 | ReAct trajectory generation (Qwen3.6-27B-AWQ) | ~1-2 h |
| 2 | Step-level uncertainty scoring | ~1-2 h |
| 3 | 32B judge error annotation | ~1-2 h |
| 4 | Localization scoring | < 1 min |
| 5 | 18 repair strategies | ~2-3 h |
| 6 | Tables, stats, figures | < 1 min |

**Every stage is resumable** — if Colab disconnects, re-run the cell and it picks up where it left off.

---
## 0. Setup

In [ ]:
# Verify GPU — should show A100
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_DIR = '/content/agent-repair'
CONFIG = 'config/config_colab.yaml'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/kishormorol/agent-repair.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Install vLLM with CUDA 12.x (Colab's CUDA version)
!pip install -q vllm --extra-index-url https://download.pytorch.org/whl/cu129 2>&1 | tail -5

# Install remaining dependencies
!pip install -q transformers>=4.57.0 tokenizers>=0.21.0 accelerate \
    huggingface_hub datasets scipy scikit-learn pandas pyarrow \
    pyyaml tqdm matplotlib seaborn statsmodels 2>&1 | tail -3

print('\n--- Installation complete ---')

In [ ]:
# Quick sanity check: can vLLM see the GPU?
import torch, vllm
print(f'torch:  {torch.__version__}')
print(f'vLLM:   {vllm.__version__}')
print(f'CUDA:   {torch.version.cuda}')
print(f'GPU:    {torch.cuda.get_device_name(0)}')
print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

---
## Smoke Test (20 questions, ~10-15 min)

**Run this first** to verify everything works before committing A100 units to the full 500-question run.

In [ ]:
stages = ['run_setup', 'run_generate', 'run_uncertainty', 'run_annotate',
          'run_localize', 'run_repair', 'run_eval']

for stage in stages:
    print(f'\n{"="*60}\n  {stage}\n{"="*60}')
    !python scripts/{stage}.py --config {CONFIG} --limit 20
    print()

In [ ]:
# Check smoke test produced output
import glob
tables = glob.glob('outputs/tables/*') + glob.glob('/content/drive/MyDrive/agent-repair/outputs/tables/*')
figs = glob.glob('outputs/figures/*') + glob.glob('/content/drive/MyDrive/agent-repair/outputs/figures/*')
print(f'Tables: {len(tables)}, Figures: {len(figs)}')
if tables or figs:
    print('Smoke test PASSED — safe to run the full pipeline below.')
else:
    print('WARNING: No outputs found. Check the logs above for errors.')

---
## Full Pipeline (500 questions)

Only run these cells after the smoke test passes. To reset and run fresh, delete the `data/processed/` folder.

### Stage 0: Download HotpotQA

In [ ]:
!python scripts/run_setup.py --config {CONFIG}

### Stage 1: Generate ReAct Trajectories (Qwen3.6-27B-AWQ)

In [ ]:
!python scripts/run_generate.py --config {CONFIG}

### Stage 2: Step-Level Uncertainty

In [ ]:
!python scripts/run_uncertainty.py --config {CONFIG}

### Stage 3: Error Annotation (32B Judge)

In [ ]:
!python scripts/run_annotate.py --config {CONFIG}

### Stage 4: Localization Scoring

In [ ]:
!python scripts/run_localize.py --config {CONFIG}

### Stage 5: Repair (18 Strategies)

In [ ]:
!python scripts/run_repair.py --config {CONFIG}

### Stage 6: Evaluation

In [ ]:
!python scripts/run_eval.py --config {CONFIG}

---
## Results

In [ ]:
from src.utils import load_config
cfg = load_config(CONFIG)

summary_path = os.path.join(cfg.path('tables'), 'summary.txt')
if os.path.exists(summary_path):
    with open(summary_path) as f:
        print(f.read())
else:
    print('Summary not yet generated. Run Stage 6 first.')

In [ ]:
import pandas as pd

results_path = os.path.join(cfg.path('tables'), 'main_results_readable.csv')
if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    display(df)
else:
    print('Results table not yet generated. Run Stage 6 first.')

In [ ]:
import glob
from IPython.display import display, Image

fig_dir = cfg.path('figures')
figs = sorted(glob.glob(os.path.join(fig_dir, '*.png')))
if figs:
    for f in figs:
        print(f'\n--- {os.path.basename(f)} ---')
        display(Image(filename=f, width=800))
else:
    print('No figures yet. Run Stage 6 first.')